# English -> Fake Transfer on Colab or VSCode+Colab

This notebook supports three project-loading modes:

- reuse an existing project directory already present in the runtime
- clone from GitHub
- unpack a local zip or tar archive into the runtime

After the project is available, it runs the full research pipeline:

1. install dependencies
2. detect the runtime and locate the project
3. load Hugging Face credentials
4. build the synthetic dataset
5. create train/dev/test splits
6. format SFT data
7. train QLoRA on a small instruct model that is easier to run on Colab
8. run inference on English and Fake questions
9. evaluate source learning and transfer


## Requirements

- A Colab GPU runtime with CUDA available
- A public instruct model or a gated model you can access
- `HF_TOKEN` only if your chosen model requires it
- The project must be present in the runtime, clonable, or uploadable as an archive

Recommended when you do not use GitHub:

1. zip the whole project folder locally
2. upload that archive into the Colab runtime or Google Drive
3. set `PROJECT_ARCHIVE_PATH` below


In [ ]:
REPO_URL = ""
REPO_BRANCH = "main"
PROJECT_DIR = "/content/MLLMs_Final_Project"
EXISTING_PROJECT_DIR = ""
PROJECT_ARCHIVE_PATH = ""
ARCHIVE_EXTRACT_ROOT = "/content"
BASE_MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
RUN_NAME = "main_qlora"
HF_TOKEN = ""
USE_DRIVE = False
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/english_fake_transfer"

NUM_SUBJECTS = 50
NUM_EPOCHS = 6
BATCH_SIZE = 1
GRAD_ACCUM = 16
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 192
GEN_BATCH_SIZE = 2
MAX_NEW_TOKENS = 24


In [ ]:
!pip install -q -U transformers peft datasets accelerate bitsandbytes pyyaml tqdm huggingface_hub sentencepiece

In [ ]:
from pathlib import Path
import zipfile

archive = Path(PROJECT_ARCHIVE_PATH)
extract_root = Path(ARCHIVE_EXTRACT_ROOT)
extract_root.mkdir(parents=True, exist_ok=True)

print("archive exists:", archive.exists())
print("is_zipfile:", zipfile.is_zipfile(archive))
if archive.exists() and zipfile.is_zipfile(archive):
    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(extract_root)

print("top-level after extract:")
for p in sorted(extract_root.iterdir()):
    print(" -", p)

print("\npossible project markers:")
for p in extract_root.rglob("train_lora.py"):
    print("train_lora.py:", p)
for p in extract_root.rglob("dataset.yaml"):
    print("dataset.yaml:", p)


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

def directory_looks_like_project(path: Path) -> bool:
    return (
        path.exists()
        and (path / 'src').exists()
        and (path / 'configs').exists()
        and (path / 'src' / 'train_lora.py').exists()
    )

def detect_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def find_project(candidates):
    for candidate in candidates:
        if directory_looks_like_project(candidate):
            return candidate.resolve()
    return None

IS_COLAB = detect_colab()
IS_VSCODE = bool(os.environ.get('VSCODE_PID') or os.environ.get('TERM_PROGRAM') == 'vscode')

candidate_paths = []
if EXISTING_PROJECT_DIR.strip():
    candidate_paths.append(Path(EXISTING_PROJECT_DIR).expanduser())
candidate_paths.append(Path.cwd())
candidate_paths.append(Path(PROJECT_DIR))

project_dir = find_project(candidate_paths)

if project_dir is None and PROJECT_ARCHIVE_PATH.strip():
    archive_path = Path(PROJECT_ARCHIVE_PATH).expanduser()
    if not archive_path.exists():
        raise FileNotFoundError(f'PROJECT_ARCHIVE_PATH does not exist: {archive_path}')
    extract_root = Path(ARCHIVE_EXTRACT_ROOT).expanduser()
    extract_root.mkdir(parents=True, exist_ok=True)
    import zipfile
    if not zipfile.is_zipfile(archive_path):
        raise ValueError(f'PROJECT_ARCHIVE_PATH is not a valid zip file: {archive_path}')
    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(extract_root)
    post_extract_candidates = candidate_paths + list(extract_root.iterdir())
    project_dir = find_project(post_extract_candidates)

if project_dir is None and REPO_URL.strip():
    clone_target = Path(PROJECT_DIR)
    if clone_target.exists() and any(clone_target.iterdir()) and not (clone_target / '.git').exists():
        raise ValueError(
            f'Clone target {clone_target} already exists and is not a git repo. '
            'Choose a different PROJECT_DIR or clean the directory first.'
        )
    if (clone_target / '.git').exists():
        subprocess.run(['git', '-C', str(clone_target), 'fetch', '--all'], check=True)
        subprocess.run(['git', '-C', str(clone_target), 'checkout', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(clone_target), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(clone_target)], check=True)
    project_dir = clone_target.resolve()

if project_dir is None:
    raise ValueError(
        'Project directory was not found in the runtime. '
        'Set EXISTING_PROJECT_DIR, or set PROJECT_ARCHIVE_PATH to a zip/tar file, '
        'or provide REPO_URL so the notebook can clone the repo.'
    )

os.chdir(project_dir)
sys.path.insert(0, str(project_dir / 'src'))

print({'is_colab': IS_COLAB, 'is_vscode': IS_VSCODE, 'project_dir': str(project_dir)})

In [ ]:
import subprocess
import torch

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    print('nvidia-smi not available or failed:', exc)

print({'cuda_available': torch.cuda.is_available(), 'device_count': torch.cuda.device_count()})
if not torch.cuda.is_available():
    raise RuntimeError('This notebook is configured for QLoRA and requires a CUDA GPU runtime.')

In [ ]:
import os
from huggingface_hub import login

hf_token = HF_TOKEN.strip() or os.environ.get('HF_TOKEN', '').strip()
if not hf_token and IS_COLAB:
    try:
        from google.colab import userdata  # type: ignore
        hf_token = (userdata.get('HF_TOKEN') or '').strip()
    except Exception:
        pass

if hf_token:
    login(token=hf_token)
    os.environ['HF_TOKEN'] = hf_token
    print('Hugging Face login complete.')
else:
    print('HF_TOKEN not provided. Continuing without login for public models.')

In [ ]:
from pathlib import Path

drive_output_root = None
if USE_DRIVE and IS_COLAB:
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive')
        drive_output_root = Path(DRIVE_OUTPUT_ROOT) / RUN_NAME
        drive_output_root.mkdir(parents=True, exist_ok=True)
        print('Drive output root:', drive_output_root)
    except Exception as exc:
        print('Drive mount skipped:', exc)
else:
    print('Drive persistence disabled or google.colab unavailable.')

In [ ]:
from pathlib import Path
import yaml

generated_dir = project_dir / 'configs' / 'generated'
generated_dir.mkdir(parents=True, exist_ok=True)


def load_yaml(path: Path):
    with path.open('r', encoding='utf-8') as handle:
        return yaml.safe_load(handle)


def dump_yaml(path: Path, data):
    with path.open('w', encoding='utf-8') as handle:
        yaml.safe_dump(data, handle, sort_keys=False, allow_unicode=True)


def build_inference_config(base_cfg, *, run_name: str, checkpoint_dir: Path, predictions_dir: Path):
    cfg = dict(base_cfg)
    cfg['experiment_name'] = run_name
    cfg['model'] = dict(base_cfg['model'])
    cfg['model']['base_model_name_or_path'] = BASE_MODEL_NAME
    cfg['model']['adapter_path'] = str(checkpoint_dir)
    cfg['output'] = dict(base_cfg['output'])
    cfg['output']['predictions_dir'] = str(predictions_dir)
    cfg['output']['run_name'] = 'default'
    cfg['generation'] = dict(base_cfg['generation'])
    cfg['generation']['batch_size'] = int(GEN_BATCH_SIZE)
    cfg['generation']['max_new_tokens'] = int(MAX_NEW_TOKENS)
    return cfg


def build_evaluate_config(base_cfg, *, run_name: str, predictions_dir: Path, metrics_dir: Path, adapter_path: Path, train_language_mode: str):
    cfg = dict(base_cfg)
    cfg['experiment_name'] = run_name
    cfg['input'] = dict(base_cfg['input'])
    cfg['input']['predictions_dir'] = str(predictions_dir / 'default')
    cfg['output'] = dict(base_cfg['output'])
    cfg['output']['metrics_dir'] = str(metrics_dir / 'default')
    cfg['metadata'] = dict(base_cfg['metadata'])
    cfg['metadata']['train_language_mode'] = train_language_mode
    cfg['metadata']['model_name'] = BASE_MODEL_NAME
    cfg['metadata']['adapter_path'] = str(adapter_path)
    return cfg


dataset_cfg = load_yaml(project_dir / 'configs' / 'dataset.yaml')
train_main_cfg = load_yaml(project_dir / 'configs' / 'train_main.yaml')
train_fake_cfg = load_yaml(project_dir / 'configs' / 'train_fake_control.yaml')
inference_base_cfg = load_yaml(project_dir / 'configs' / 'inference.yaml')
evaluate_base_cfg = load_yaml(project_dir / 'configs' / 'evaluate.yaml')

dataset_cfg['dataset']['num_subjects'] = int(NUM_SUBJECTS)

for train_cfg, train_language_mode, run_name in [
    (train_main_cfg, 'en_only', RUN_NAME),
    (train_fake_cfg, 'en_plus_fake', f'{RUN_NAME}_fake_control'),
]:
    train_cfg['experiment_name'] = run_name
    train_cfg['training_mode'] = 'qlora'
    train_cfg['model']['name_or_path'] = BASE_MODEL_NAME
    train_cfg['quantization']['enabled'] = True
    train_cfg['training']['num_train_epochs'] = int(NUM_EPOCHS)
    train_cfg['training']['per_device_train_batch_size'] = int(BATCH_SIZE)
    train_cfg['training']['gradient_accumulation_steps'] = int(GRAD_ACCUM)
    train_cfg['training']['learning_rate'] = float(LEARNING_RATE)
    train_cfg['training']['max_seq_length'] = int(MAX_SEQ_LENGTH)
    train_cfg['data']['train_language_mode'] = train_language_mode

main_run_root = Path('/content') / 'runs' / RUN_NAME
main_checkpoint_dir = main_run_root / 'checkpoints'
main_predictions_dir = main_run_root / 'predictions'
main_metrics_dir = main_run_root / 'metrics'
train_main_cfg['training']['output_dir'] = str(main_checkpoint_dir)

fake_run_root = Path('/content') / 'runs' / f'{RUN_NAME}_fake_control'
fake_checkpoint_dir = fake_run_root / 'checkpoints'
fake_predictions_dir = fake_run_root / 'predictions'
fake_metrics_dir = fake_run_root / 'metrics'
train_fake_cfg['training']['output_dir'] = str(fake_checkpoint_dir)

inference_main_cfg = build_inference_config(
    inference_base_cfg,
    run_name=RUN_NAME,
    checkpoint_dir=main_checkpoint_dir,
    predictions_dir=main_predictions_dir,
)
inference_fake_cfg = build_inference_config(
    inference_base_cfg,
    run_name=f'{RUN_NAME}_fake_control',
    checkpoint_dir=fake_checkpoint_dir,
    predictions_dir=fake_predictions_dir,
)

evaluate_main_cfg = build_evaluate_config(
    evaluate_base_cfg,
    run_name=RUN_NAME,
    predictions_dir=main_predictions_dir,
    metrics_dir=main_metrics_dir,
    adapter_path=main_checkpoint_dir,
    train_language_mode='en_only',
)
evaluate_fake_cfg = build_evaluate_config(
    evaluate_base_cfg,
    run_name=f'{RUN_NAME}_fake_control',
    predictions_dir=fake_predictions_dir,
    metrics_dir=fake_metrics_dir,
    adapter_path=fake_checkpoint_dir,
    train_language_mode='en_plus_fake',
)

paths = {
    generated_dir / 'dataset_colab.yaml': dataset_cfg,
    generated_dir / 'train_main_colab.yaml': train_main_cfg,
    generated_dir / 'inference_main_colab.yaml': inference_main_cfg,
    generated_dir / 'evaluate_main_colab.yaml': evaluate_main_cfg,
    generated_dir / 'train_fake_control_colab.yaml': train_fake_cfg,
    generated_dir / 'inference_fake_control_colab.yaml': inference_fake_cfg,
    generated_dir / 'evaluate_fake_control_colab.yaml': evaluate_fake_cfg,
}

for path, payload in paths.items():
    dump_yaml(path, payload)
    print('wrote', path)


In [ ]:
!python src/build_dataset.py --config configs/generated/dataset_colab.yaml
!python src/make_splits.py --config configs/generated/dataset_colab.yaml
!python src/format_for_sft.py --config configs/generated/dataset_colab.yaml


In [ ]:
!python src/train_lora.py --config configs/generated/train_main_colab.yaml


In [ ]:
!python src/inference.py --config configs/generated/inference_main_colab.yaml


In [ ]:
!python src/evaluate.py --config configs/generated/evaluate_main_colab.yaml


In [ ]:
from pathlib import Path
import json

metrics_path = Path('/content') / 'runs' / RUN_NAME / 'metrics' / 'default' / 'metrics.json'
with metrics_path.open('r', encoding='utf-8') as handle:
    metrics = json.load(handle)
metrics


## Optional: Fake comprehension control
Run this after the main experiment if you want the control model on the same first-version setup.


In [ ]:
!python src/train_lora.py --config configs/generated/train_fake_control_colab.yaml


In [ ]:
!python src/inference.py --config configs/generated/inference_fake_control_colab.yaml


In [ ]:
!python src/evaluate.py --config configs/generated/evaluate_fake_control_colab.yaml


In [ ]:
from pathlib import Path
import json

metrics_path = Path('/content') / 'runs' / f'{RUN_NAME}_fake_control' / 'metrics' / 'default' / 'metrics.json'
with metrics_path.open('r', encoding='utf-8') as handle:
    metrics = json.load(handle)
metrics


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil

run_roots = [
    Path('/content') / 'runs' / RUN_NAME,
    Path('/content') / 'runs' / f'{RUN_NAME}_fake_control',
]
drive_root = Path(DRIVE_OUTPUT_ROOT)

if USE_DRIVE:
    drive_root.mkdir(parents=True, exist_ok=True)
    for src in run_roots:
        if not src.exists():
            print('missing:', src)
            continue
        dst = drive_root / src.name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print('copied:', src, '->', dst)
else:
    print('USE_DRIVE is False; skipping copy.')


In [ ]:
from pathlib import Path
import shutil

bundle_root = Path('/content/english_fake_transfer_bundle')
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir(parents=True, exist_ok=True)

copy_items = [
    (Path('/content') / 'runs' / RUN_NAME, bundle_root / RUN_NAME),
    (Path('/content') / 'runs' / f'{RUN_NAME}_fake_control', bundle_root / f'{RUN_NAME}_fake_control'),
    (Path('/content/MLLMs_Final_Project') / 'outputs' / 'analysis', bundle_root / 'analysis'),
]

for src, dst in copy_items:
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print('copied:', src, '->', dst)
    else:
        print('missing:', src)

zip_base = '/content/english_fake_transfer_bundle'
zip_file = shutil.make_archive(zip_base, 'zip', root_dir=bundle_root.parent, base_dir=bundle_root.name)
print('zip created:', zip_file)
print('size_mb:', round(Path(zip_file).stat().st_size / 1024 / 1024, 2))


In [ ]:
from google.colab import files
files.download('/content/english_fake_transfer_bundle.zip')
